# Physics prompting on Google Colab — P1-VL-30B-A3B (Q4_K_M)

Runs the questions with one copy of the model and saves each answer to Google Drive:
- `regular_questions.json` → regular prompt only, e.g. `Q1_regular_regular_prompt.md`
- `obvious_questions.json` → both prompts, e.g. `Q1_obvious_regular_prompt.md`, `Q1_obvious_detailed_prompt.md`
- `non_obvious_questions.json` → both prompts, e.g. `Q1_non_obvious_detailed_prompt.md`

plus `log.jsonl` with every run.

**Before the first run:**
1. **Runtime → Change runtime type → A100 GPU** (or L4, or H100 if it's listed). T4 is too small.
2. Click the **key icon** (Secrets) in the left sidebar and add `GITHUB_TOKEN` (your GitHub personal access token). Turn on **Notebook access**.
   Optional: also add `HF_TOKEN` (Hugging Face Read token) for a faster download.

Then **Runtime → Run all**. If the session disconnects, reconnect and **Run all** again: finished answers are skipped.

## 1. Settings

In [ ]:
GITHUB_USER = "Himson2006"
GITHUB_REPO = "physics-prompting"
DRIVE_DIR = "/content/drive/MyDrive/physics-prompting"   # answers go to DRIVE_DIR/outputs

RUN_SETS = None          # None = all three sets; or e.g. ["obvious", "non_obvious"]
RUN_IDS = None           # None = every question in each set; or e.g. ["Q1"] for a quick test
SEED = 0                 # keep the same for every run you want to compare
GEN_KWARGS = {}          # e.g. {"max_tokens": 28000} if answers are truncated
SKIP_EXISTING = True     # skip question+prompt pairs that already have a file; False = run again (saved as _2, _3, ...)

## 2. Check the GPU

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used",
                    "--format=csv,noheader,nounits"], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError("No GPU. Runtime → Change runtime type → choose A100 or L4.")
name, total, used = [x.strip() for x in r.stdout.splitlines()[0].split(",")]
free_gib = (int(total) - int(used)) / 1024
print(f"GPU: {name}, {free_gib:.1f} GiB free")
if free_gib < 21:
    raise RuntimeError(f"{name} has {free_gib:.1f} GiB free; the model needs ~21 GiB. "
                       "Switch to an A100 or L4 runtime.")

## 3. Connect Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
OUTPUT_DIR = f"{DRIVE_DIR}/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Answers will be saved to", OUTPUT_DIR)

## 4. Get the latest code and questions from GitHub

In [ ]:
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
REPO_DIR = f"/content/{GITHUB_REPO}"
if os.path.isdir(REPO_DIR):
    cmd = ["git", "-C", REPO_DIR, "pull"]
else:
    cmd = ["git", "clone", f"https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git", REPO_DIR]
r = subprocess.run(cmd, capture_output=True, text=True)
print((r.stdout + r.stderr).replace(token, "***"))
if r.returncode != 0:
    raise RuntimeError("git failed. Check the GITHUB_TOKEN secret (repo access, Contents: read).")

## 5. Install llama-cpp-python with GPU support
Tries, in order: a copy saved on your Drive from an earlier session → the prebuilt CUDA package (~1 min) →
building from source (~15–25 min, then saved to Drive so later sessions are fast).

In [ ]:
import glob, sys
ip = get_ipython()
WHEEL_DIR = f"{DRIVE_DIR}/wheels"
VERSION = "0.3.19"

def gpu_build_ok():
    r = subprocess.run([sys.executable, "-c",
                        "import llama_cpp; print(llama_cpp.llama_supports_gpu_offload())"],
                       capture_output=True, text=True)
    return r.stdout.strip().endswith("True")

if not gpu_build_ok():
    cached = sorted(glob.glob(f"{WHEEL_DIR}/llama_cpp_python-{VERSION}-*.whl"))
    if cached:
        print("Installing saved build from Drive...")
        ip.system(f'pip install -q --no-deps --force-reinstall "{cached[-1]}"')
if not gpu_build_ok():
    print("Trying the prebuilt CUDA package...")
    ip.system(f"pip install -q --only-binary llama-cpp-python llama-cpp-python=={VERSION} "
              "--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124")
if not gpu_build_ok():
    print("Prebuilt package didn't work here; building from source (15-25 min)...")
    os.makedirs(WHEEL_DIR, exist_ok=True)
    ip.system(f'CMAKE_ARGS="-DGGML_CUDA=on" pip wheel --no-deps llama-cpp-python=={VERSION} -w "{WHEEL_DIR}"')
    built = sorted(glob.glob(f"{WHEEL_DIR}/llama_cpp_python-{VERSION}-*.whl"))
    ip.system(f'pip install -q --no-deps --force-reinstall "{built[-1]}"')

assert gpu_build_ok(), "llama-cpp-python still has no GPU support; see the messages above."
print("llama-cpp-python", VERSION, "with GPU support is ready.")

## 6. Load the model
The first time in each session this downloads ~19 GB (a few minutes).

In [ ]:
os.environ["PHYSICS_OUTPUTS_DIR"] = OUTPUT_DIR
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
import pipeline
pipeline.load_model()
for qs, prompts in pipeline.QUESTION_SETS.items():
    print(f"{qs}: {list(pipeline.load_questions(qs))} → {prompts}")

## 7. Run
Each line shows progress, then the answer. Keep this browser tab open while it runs.

In [ ]:
pipeline.run_batch(RUN_SETS, RUN_IDS, seed=SEED, gen_kwargs=GEN_KWARGS, skip_existing=SKIP_EXISTING)

## 8. Check the results
The files are in Google Drive at `MyDrive/physics-prompting/outputs/`. This lists them and flags truncated answers.

In [ ]:
import json
rows = [json.loads(l) for l in open(f"{OUTPUT_DIR}/log.jsonl", encoding="utf-8")] if os.path.exists(f"{OUTPUT_DIR}/log.jsonl") else []
for f in sorted(glob.glob(f"{OUTPUT_DIR}/*.md")):
    print(os.path.basename(f))
bad = [r.get("file", r["question_id"]) for r in rows if r["truncated"]]
print(f"\n{len(rows)} runs logged; truncated: {bad or 'none'}")

## 9. Optional: also push the answers to GitHub

In [ ]:
PUSH_TO_GITHUB = False

if PUSH_TO_GITHUB:
    ip.system(f'mkdir -p "{REPO_DIR}/outputs" && cp "{OUTPUT_DIR}"/*.md "{OUTPUT_DIR}"/log.jsonl "{REPO_DIR}/outputs/"')
    for cmd in (["git", "config", "user.name", "Himson Chapagain"],
                ["git", "config", "user.email", "chapagain.himson00@gmail.com"],
                ["git", "add", "outputs"],
                ["git", "commit", "-m", "Results from Colab"],
                ["git", "push"]):
        r = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
        print((r.stdout + r.stderr).replace(token, "***"))